# 12 – Faltungsnetz (CNN): Bilder wie ein Bildverarbeitungssystem auswerten

**Lernsituation „KI-Sichtprüfstation“ · Doppelstunden 6 – 7**

### Lernziele
Du kannst …
- die Faltung (Convolution) und das Pooling erklären und selbst in NumPy berechnen,
- ein CNN mit Keras aufbauen und seine Parameterzahl begründen,
- ein Modell mit Konfusionsmatrix, Precision und Recall bewerten,
- eine Konfidenzschwelle für die Prüfstation festlegen und ihre Folgen abschätzen,
- ein Modell für den Einsatz auf dem Raspberry Pi exportieren.

| Teil | Doppelstunde | Inhalt |
|---|:--:|---|
| A | 6 | Faltung und Pooling verstehen, selbst implementieren, CNN aufbauen |
| B | 7 | Training, Bewertung, Fehlerkosten, Konfidenzschwelle, Export · 🧪 Teilzielkontrolle 3 |

| Symbol | Bedeutung |
|:--:|---|
| 📖 | Grundlagen lesen / interaktiv erkunden |
| ✏️ | Aufgabe: Code **selbst schreiben** (Zellen mit `# TODO`) |
| 🔍 | Experiment / Beobachtung – Ergebnisse notieren |
| 🧪 | Teilzielkontrolle |
| 🏭 | Bezug zur industriellen Fertigung |
| 🍓 | Hinweis für den Raspberry Pi |

In [ ]:
# =============================================================================
# Vorbereitung: Hilfsmodul finden (klappt im Hauptordner und in loesungen/)
# =============================================================================
import sys
from pathlib import Path

ORDNER = Path.cwd() if (Path.cwd() / "ki_hilfen.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ORDNER))

import numpy as np
import cv2
import matplotlib.pyplot as plt
import ki_hilfen as kh

import time
import keras
from keras import layers
import tensorflow as tf

kh.setze_seed(42)

---
## Teil A – Faltung und Pooling (Doppelstunde 6)

### 📖 Interaktiv: Wie ein Filter Kanten findet

In [ ]:
kh.zeige_html("06_faltung.html", hoehe=1100)

### ✏️ Aufgabe A1 – Faltung selbst programmieren
Schreibe die Funktion `falte(bild, kernel)` für ein Graustufenbild und einen 3 × 3-Kernel **ohne Padding** („valid“):
- Das Ergebnis ist 2 Pixel schmaler und 2 Pixel niedriger als das Bild.
- Für jede Position `(i, j)`: Ausschnitt `bild[i:i+3, j:j+3]` mit dem Kernel elementweise multiplizieren und alles aufsummieren.

In [ ]:
def falte(bild, kernel):
    h, w = bild.shape
    k = kernel.shape[0]
    ergebnis = np.zeros((h - k + 1, w - k + 1), dtype="float32")
    # TODO: zwei Schleifen über i und j, Ausschnitt * kernel aufsummieren
    return ergebnis


# Selbstkontrolle mit einem kleinen Beispiel
probe = np.array([[0, 0, 1, 1],
                  [0, 0, 1, 1],
                  [0, 0, 1, 1]], dtype="float32")
kante_senkrecht = np.array([[-1, 0, 1],
                            [-1, 0, 1],
                            [-1, 0, 1]], dtype="float32")
print(falte(probe, kante_senkrecht))
kh.pruefe(lambda: np.allclose(falte(probe, kante_senkrecht), [[3, 3]]), "Faltung richtig berechnet",
          "Für das Beispiel sollte [[3, 3]] herauskommen")

Jetzt auf ein echtes Bild – mit drei verschiedenen Filtern:

In [ ]:
DATENSATZ = ORDNER / "dataset"
datei = kh.bilddateien(DATENSATZ / "3")[5]                       # ein Quadrat
grau = kh.bild_vorbereiten(cv2.imread(str(datei)), 96, "grau")[..., 0]
grau = 1 - grau.astype("float32") / 255                          # Strich = hell (1), Papier = dunkel (0)

filter_ = {
    "senkrechte Kanten": kante_senkrecht,
    "waagrechte Kanten": kante_senkrecht.T,
    "Diagonale ↘": np.array([[2, -1, -1], [-1, 2, -1], [-1, -1, 2]], dtype="float32"),
}
fig, ax = plt.subplots(1, 4, figsize=(15, 4))
ax[0].imshow(grau, cmap="gray"); ax[0].set_title("Eingabe 96 × 96")
for a, (name, k) in zip(ax[1:], filter_.items()):
    karte = np.maximum(falte(grau, k), 0)                         # ReLU
    a.imshow(karte, cmap="magma"); a.set_title(f"{name}\n{karte.shape[0]} × {karte.shape[1]}")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

### ✏️ Aufgabe A2 – Max-Pooling selbst programmieren
`max_pool(karte)` soll die Merkmalskarte in Blöcke von 2 × 2 Pixeln zerlegen und aus jedem Block nur den **größten** Wert behalten. Eine ungerade Zeile/Spalte am Rand wird weggelassen.

In [ ]:
def max_pool(karte):
    h, w = karte.shape[0] // 2, karte.shape[1] // 2
    ergebnis = np.zeros((h, w), dtype=karte.dtype)
    # TODO
    return ergebnis


probe = np.array([[1, 2, 0, 0],
                  [3, 4, 0, 5],
                  [0, 0, 7, 0],
                  [1, 0, 0, 0]], dtype="float32")
print(max_pool(probe))
kh.pruefe(lambda: np.array_equal(max_pool(probe), [[4, 5], [1, 7]]), "Max-Pooling richtig",
          "Erwartet wird [[4, 5], [1, 7]]")
kh.pruefe(lambda: max_pool(np.zeros((5, 7))).shape == (2, 3), "Ungerade Größen werden abgerundet",
          "Bei einer 5 × 7-Karte sollte das Ergebnis 2 × 3 sein")

### 📖 Ein CNN in Keras
In einem CNN werden die Kernelwerte **nicht vorgegeben, sondern gelernt**. Jede `Conv2D`-Schicht hat viele Filter; ihre Merkmalskarten werden gestapelt und an die nächste Schicht weitergegeben. Je tiefer die Schicht, desto komplexer die Muster.

![CNN](material/bilder/cnn.svg)

🍓 **Bildgröße auf dem Raspberry Pi:** 96 × 96 ist ein guter Kompromiss (Pi 5: ca. 10 – 20 s pro Epoche). Auf einem Pi 4 besser `GROESSE = 64` wählen. Auf einem PC mit Grafikkarte sind auch 224 × 224 möglich.

In [ ]:
GROESSE = 96
FARBE = "rgb"
BATCH = 32
EPOCHEN = 80
LERNRATE = 1e-3

X, y, klassen = kh.lade_datensatz(DATENSATZ, GROESSE, FARBE)
X_train, X_val, X_test, y_train, y_val, y_test = kh.teile_datensatz(X, y)
ANZAHL_KLASSEN = len(klassen)
EINGABE = X_train.shape[1:]
print("Eingabe:", EINGABE, "| Training", len(X_train), "· Validierung", len(X_val), "· Test", len(X_test))

augmentierung = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2),
], name="augmentierung")

train_ds = kh.als_tf_dataset(X_train, y_train, BATCH, mischen=True, augmentierung=augmentierung)
val_ds = kh.als_tf_dataset(X_val, y_val, BATCH)
test_ds = kh.als_tf_dataset(X_test, y_test, BATCH)

### ✏️ Aufgabe A3 – Das CNN aufbauen
Ergänze das Modell. Der erste Block ist vorgegeben.

| Block | Schichten |
|---|---|
| Eingang | `Input`, `Rescaling(1/255)` *(vorgegeben)* |
| 1 | `Conv2D(16, 3, padding="same", activation="relu")`, `MaxPooling2D()` *(vorgegeben)* |
| 2 | `Conv2D(32, …)`, `MaxPooling2D()` |
| 3 | `Conv2D(64, …)`, `MaxPooling2D()` |
| 4 | `Conv2D(128, …)` |
| Kopf | `GlobalAveragePooling2D()`, `Dropout(0.3)`, `Dense(ANZAHL_KLASSEN, activation="softmax")` |

📖 `GlobalAveragePooling2D` bildet von jeder Merkmalskarte den Mittelwert – aus 128 Karten werden 128 Zahlen. Das spart gegenüber `Flatten` sehr viele Parameter.

In [ ]:
def baue_cnn():
    return keras.Sequential([
        keras.Input(shape=EINGABE),
        layers.Rescaling(1 / 255),
        layers.Conv2D(16, 3, padding="same", activation="relu", name="conv1"),
        layers.MaxPooling2D(),
        # TODO: Blöcke 2 bis 4 und den Kopf ergänzen
    ], name="cnn")


modell = baue_cnn()
modell.summary()
kh.pruefe(lambda: modell.output_shape == (None, ANZAHL_KLASSEN), "Ausgabe passt zur Klassenzahl",
          "Die letzte Schicht braucht ANZAHL_KLASSEN Neuronen mit Softmax")
kh.pruefe(lambda: sum(type(s).__name__ == "Conv2D" for s in modell.layers) == 4, "Vier Faltungsschichten",
          "Es sollten vier Conv2D-Schichten sein")
kh.pruefe(lambda: modell.count_params() < 150_000, f"Kompakt: {modell.count_params():,} Parameter",
          "Mehr als 150 000 Parameter – hast du Flatten statt GlobalAveragePooling2D verwendet?")

🔍 **Rechne nach:** Die Schicht `conv2` hat 4 640 Parameter. Erkläre die Zahl (Tipp: 32 Filter, Kernel 3 × 3, 16 Eingangskanäle, Bias). Vergleiche die Gesamtzahl der Parameter mit dem FCNN aus Notebook 11 – obwohl das CNN 9-mal größere Bilder in Farbe verarbeitet!

---
## Teil B – Trainieren und bewerten (Doppelstunde 7)

In [ ]:
modell.compile(optimizer=keras.optimizers.Adam(LERNRATE),
               loss="sparse_categorical_crossentropy", metrics=["accuracy"])

pfad_bestes = ORDNER / "modelle" / "cnn_bestes.weights.h5"
pfad_bestes.parent.mkdir(exist_ok=True)
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=15),
    keras.callbacks.ModelCheckpoint(pfad_bestes, monitor="val_loss",
                                    save_best_only=True, save_weights_only=True),
]

start = time.time()
verlauf = modell.fit(train_ds, validation_data=val_ds, epochs=EPOCHEN,
                     callbacks=callbacks, verbose=2)
print(f"Trainingszeit: {(time.time() - start) / 60:.1f} min")
modell.load_weights(pfad_bestes)
kh.zeige_lernkurven(verlauf)

### 🔍 Was hat das CNN gelernt?
Die Merkmalskarten der ersten Faltungsschicht für ein Testbild – vergleiche mit deinen selbst gerechneten Kantenbildern aus Aufgabe A1.

In [ ]:
schicht = "conv1"
merkmale_modell = keras.Model(modell.inputs, modell.get_layer(schicht).output)
karten = merkmale_modell.predict(X_test[:1].astype("float32"), verbose=0)[0]

plt.figure(figsize=(14, 4))
plt.subplot(2, 9, 1); plt.imshow(X_test[0]); plt.title("Eingabe"); plt.axis("off")
for i in range(min(16, karten.shape[-1])):
    plt.subplot(2, 9, i + 2 + (i >= 8)); plt.imshow(karten[..., i], cmap="magma"); plt.axis("off")
plt.suptitle(f"16 Merkmalskarten der Schicht {schicht}"); plt.tight_layout(); plt.show()

### 📖 Interaktiv: Konfusionsmatrix und Fehlerkosten

In [ ]:
kh.zeige_html("07_bewertung.html", hoehe=1300)

In [ ]:
_, test_acc = modell.evaluate(test_ds, verbose=0)
print(f"Test-Genauigkeit CNN: {test_acc:.1%}")

wahrsch = modell.predict(X_test.astype("float32"), verbose=0)
y_pred = wahrsch.argmax(axis=1)
cm = kh.zeige_konfusionsmatrix(y_test, y_pred, klassen, titel="CNN – Testdaten")

### ✏️ Aufgabe B1 – Die häufigste Verwechslung finden
Finde in der Konfusionsmatrix `cm` den größten Wert **außerhalb der Diagonale** und gib aus, welche Klasse wie oft für welche andere gehalten wurde. Tipp: Kopie anlegen, Diagonale mit `np.fill_diagonal(kopie, 0)` auf 0 setzen, dann `np.unravel_index(kopie.argmax(), kopie.shape)`.

In [ ]:
wahr_idx, vorh_idx = None, None   # TODO

print(f"Häufigste Verwechslung: {klassen[wahr_idx]} wurde {cm[wahr_idx, vorh_idx]}× als {klassen[vorh_idx]} erkannt")
ohne_diag = cm - np.diag(np.diag(cm))
kh.pruefe(lambda: cm[wahr_idx, vorh_idx] == ohne_diag.max() and wahr_idx != vorh_idx,
          "Größte Verwechslung gefunden", "Gesucht ist der größte Wert außerhalb der Diagonale")

### 🏭 Konfidenzschwelle: Wann soll die Station selbst entscheiden?
Die höchste Softmax-Wahrscheinlichkeit nennen wir **Konfidenz**. Ist sie zu niedrig, soll die Station das Teil **nicht** automatisch einsortieren, sondern zur manuellen Prüfung ausschleusen (Rückweisung).

### ✏️ Aufgabe B2 – Wirkung einer Schwelle berechnen
Berechne für die Testdaten und eine Schwelle `schwelle`:
- `konfidenz`: die höchste Wahrscheinlichkeit je Bild,
- `angenommen`: boolesches Array – `True`, wenn `konfidenz >= schwelle`,
- `anteil_rueckweisung`: Anteil der Bilder, die zur manuellen Prüfung gehen,
- `genauigkeit_angenommen`: Genauigkeit nur auf den angenommenen Bildern.

In [ ]:
schwelle = 0.80

konfidenz = None              # TODO
angenommen = None             # TODO
anteil_rueckweisung = None    # TODO
genauigkeit_angenommen = None # TODO

print(f"Schwelle {schwelle:.0%}: {anteil_rueckweisung:.1%} zur manuellen Prüfung, "
      f"Genauigkeit der automatischen Entscheidungen {genauigkeit_angenommen:.1%} "
      f"(ohne Schwelle {test_acc:.1%})")
kh.pruefe(lambda: np.allclose(konfidenz, wahrsch.max(axis=1)), "Konfidenz richtig",
          "konfidenz = höchste Wahrscheinlichkeit je Bild")
kh.pruefe(lambda: abs(anteil_rueckweisung - (wahrsch.max(axis=1) < schwelle).mean()) < 1e-9,
          "Rückweisungsanteil richtig", "Anteil der Bilder mit konfidenz < schwelle")

In [ ]:
# Wie verändern sich Rückweisung und Genauigkeit mit der Schwelle?
schwellen = np.linspace(0.3, 0.99, 40)
rueck = [(wahrsch.max(1) < s).mean() for s in schwellen]
gen = [(y_pred[wahrsch.max(1) >= s] == y_test[wahrsch.max(1) >= s]).mean() if (wahrsch.max(1) >= s).any() else np.nan
       for s in schwellen]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(schwellen, np.array(gen) * 100, label="Genauigkeit angenommen")
ax[0].plot(schwellen, np.array(rueck) * 100, label="Rückweisung")
ax[0].set_xlabel("Schwelle"); ax[0].set_ylabel("%"); ax[0].grid(alpha=.3); ax[0].legend()
richtig = y_pred == y_test
ax[1].hist(wahrsch.max(1)[richtig], bins=20, alpha=.7, label="richtig erkannt", color="#2e8b57")
ax[1].hist(wahrsch.max(1)[~richtig], bins=20, alpha=.7, label="falsch erkannt", color="#c0392b")
ax[1].set_xlabel("Konfidenz"); ax[1].set_ylabel("Anzahl Testbilder"); ax[1].legend()
plt.tight_layout(); plt.show()

🔍 **Entscheidet im Team:** Welche Schwelle würdet ihr für die Prüfstation wählen? Begründet mit den Diagrammen und mit den Kosten (manuelle Prüfung vs. Fehlentscheidung).

### Modell speichern und exportieren
`speichere_modell` legt zusätzlich eine **TFLite**-Datei an. TensorFlow Lite ist für Kleinstrechner optimiert und auf dem Raspberry Pi bei der Vorhersage deutlich schneller als Keras.

In [ ]:
info = kh.speichere_modell(modell, "cnn", klassen, GROESSE, FARBE,
                           test_genauigkeit=round(float(test_acc), 4), datensatz=DATENSATZ.name)
for datei in sorted((ORDNER / "modelle").glob("cnn.*")):
    print(f"{datei.name:12s} {datei.stat().st_size / 1e6:6.2f} MB")

## 🧪 Teilzielkontrolle 3

In [ ]:
kh.zeige_html("tzk3_cnn_und_bewertung.html", hoehe=900)

### Ausblick
Unser CNN musste alles von Null lernen – aus gut 1 000 Bildern. Große Netze wie **MobileNetV2** wurden mit über einer Million Bilder vortrainiert und kennen Kanten, Ecken und Formen bereits. In **Notebook 13** nutzen wir dieses Wissen (Transfer Learning).

In [ ]:
kh.ram_aufraeumen()